# Comprehensive Fine-Mapping Framework

## Background

Our current pipeline uses [**imputed summary statistics**](https://academic.oup.com/bioinformatics/article/30/20/2906/2422225), **rss_QC**, and **BVSR**.

---

### 🔍 SuSiE_RSS

#### SuSiE (Sum of Single Effects)

SuSiE is a novel model for variable selection in linear regression, emphasizing **quantification of uncertainty** in the selection process. Key characteristics:

- Represents the sparse regression coefficient vector as a **sum of single-effect vectors**, each having only one non-zero element.
- Each iteration fits a **Single Effect Regression (SER)** model to updated residuals and updates posterior probabilities.
- The iterative procedure (IBSS) provides a **posterior distribution** over variables, helping quantify uncertainty. At every iteration, it fits a Single Effect Regression (SER) model to updated residuals, and updates its belief over which variable explains the residual best.
- Tends to output **credible sets (CS)** as thoroughly as possible.
- Fit 𝐿 effects together, and iteratively update each one by conditioning on the others until convergence.

#### SuSiE_RSS

SuSiE_RSS extends SuSiE to work with **summary statistics**, requiring an external LD matrix.

---

### 🧹 `rss_qc` (Summary Statistic Quality Control)

A preprocessing step for SuSiE_RSS to improve accuracy by identifying and correcting LD mismatches.

- **Outlier Detection**:
  - Compares observed z-scores with those expected from the LD matrix.
  - Removes SNPs with z-scores inconsistent with LD, which may signal genotyping or imputation errors.

- **LD Discrepancy Correction**:
  - Identifies variants with inflated or deflated signals due to LD mismatch.
  - Corrects or removes these variants to improve model robustness.

---

### ⚠️ Current Challenges

#### LD Mismatch

1. **rss_qc** and **SuSiE_RSS** both depend on external LD matrices.
2. SuSiE_RSS uses the LD matrix **once per iteration** when fitting SER models, meaning:
   - Inaccurate LD can lead to **compounding errors** as the algorithm proceeds adaptively.
   - Repeated use of mismatched LD data can **diverge the model** from the true signals.

---

### 🧠 Other Models in the Pipeline

- **BVSR (Bayesian Variable Selection in Regression)**: 
  - Flexible with `L = L`, `max_iter = max_iter`.
  - Fully adaptive, considers multiple effects.

- **BCR (Bayesian Conditional Regression)**: 
  - Special case of BVSR with `max_iter = 1`.

- **SER (Single Effect Regression)**:
  - Simplest case with `L = 1`, `max_iter = 1`.
  - Assumes only **one causal variant**.

---

## 📐 Framework Overview

1. Run a BVSR model
2. detect spurious BVSR results
3. create a diagnostic table with info from BVSR results and add additional analyses to retain more information  
4. made desicion based on the diagnostics info and export the results from the selected model

---

## ❓ Is Imputation Robust to LD Inaccuracies?

### ✅ Somewhat Robust — Here's Why:

1. **Regularization (`lamb`) Stabilizes Inversion**:
   - Adds a small constant to the LD matrix diagonal (`LD + λI`).
   - Helps prevent issues from noisy or nearly singular LD matrices — like **ridge regression**.

2. **Filtering Small Eigenvalues (`rcond`)**:
   - Uses a **pseudo-inverse** that discards unstable components (i.e., small eigenvalues).
   - Improves robustness by avoiding overfitting to noise.

3. **R² Thresholds Filter Weakly Imputed SNPs**:
   - SNPs with low imputation quality (low R²) are removed.
   - Prevents overreliance on **uncertain LD structures**.

---

### ⚠️ But Not Fully Immune — Limitations:

1. **Severe LD Mismatch Can Degrade Accuracy**:
   - Using mismatched LD panels (e.g., European LD on East Asian data) can lead to **biased imputation**.
   - Z-scores may become imprecise or misleading.

2. **LD Uncertainty Is Not Modeled**:
   - Assumes LD matrix is **known and fixed**.
   - No propagation of **uncertainty in LD estimation** to the final results.
   - Less robust than **Bayesian methods** that integrate over LD uncertainty.

